# 📦 SciPy Sparse Data — Beginner to Practical Notes
> **Source:** https://www.w3schools.com/python/scipy/scipy_sparse_data.php
> **Student:** Syed Tehmeed Jafar | **Date:** 2026-11-26
> **Review dates:** ✅ 2026-11-27 (quick scan) | 🔄 2026-12-01 (full revision)

---


## 1. 🤔 What is Sparse Data? (Simple Idea)

**Simple definition:** Sparse data = data where **most values are zero (or empty)**.

> Think of it like a **cinema hall** with 500 seats but only 20 people sitting.
> 480 seats are empty. If someone asks "who is sitting where?" — you only need to list 20 people, not all 500 seats.

| Type | Meaning | Example |
|------|---------|---------|
| **Sparse** | Most values = 0 | Attendance sheet (mostly absent) |
| **Dense** | Most values ≠ 0 | Exam marks sheet (everyone attempted) |

### Real-Life Examples of Sparse Data:
- **Online shopping:** 1 million products, a user buys only 5 → 999,995 zeros in their purchase row
- **Text analysis (NLP):** vocabulary of 50,000 words, one email uses only 200 words → mostly zeros
- **Hotel bookings:** 365 days, a room is booked only 30 days → 335 zeros
- **Social network:** 1 million users, you follow only 150 → massive sparse matrix


## 2. 📐 Why Do We Need a Special Format?

### The Problem:
If you store a matrix of **1 million × 1 million** where 99% values are 0:
- **Normal array** → stores all 1 trillion values → wastes **RAM** and **time**
- **Sparse format** → stores ONLY the non-zero values → saves 99% memory!

### The Solution: `scipy.sparse`
SciPy gives us special sparse matrix formats that:
- Store only **non-zero values** and their **positions**
- Support all normal matrix operations (add, multiply, reshape, etc.)
- Are used in **Machine Learning** (scikit-learn uses sparse matrices internally!)


## 3. 🔧 Setup — Import What We Need


In [1]:
import numpy as np
from scipy.sparse import csr_matrix, csc_matrix   # two main sparse types
import warnings
warnings.filterwarnings('ignore')

print("✅ scipy.sparse imported successfully!")


✅ scipy.sparse imported successfully!


## 4. 🏗️ CSR vs CSC — Two Types of Sparse Matrices

| Format | Full Name | Best For |
|--------|-----------|----------|
| **CSR** | Compressed Sparse **Row** | Fast **row** slicing, matrix-vector multiply |
| **CSC** | Compressed Sparse **Column** | Fast **column** slicing, arithmetic operations |

> **Memory trick:**
> - CSR = Row-wise → think **R**ow = CSR
> - CSC = Column-wise → think **C**olumn = CSC

**When to use which?**
- If you mostly work **row by row** (each row = one sample) → **CSR** (most ML uses this)
- If you mostly work **column by column** (each column = one feature) → **CSC**


## 5. ✍️ Creating a CSR Matrix — Step by Step


In [2]:
# Create a normal numpy array — imagine this is a user's purchase history
# Columns = Products (P1 to P9), 1 = bought, 0 = not bought
purchase_row = np.array([0, 0, 0, 0, 0, 1, 1, 0, 2])
#                        P1 P2 P3 P4 P5 P6 P7 P8 P9
# User bought: P6 (qty 1), P7 (qty 1), P9 (qty 2)
# Most values are 0 → perfect sparse data!

print("Normal array:", purchase_row)
print("Shape:", purchase_row.shape)


Normal array: [0 0 0 0 0 1 1 0 2]
Shape: (9,)


In [3]:
# Convert to CSR sparse matrix
sparse_purchase = csr_matrix(purchase_row)

print("Sparse CSR Matrix:")
print(sparse_purchase)

# It prints only NON-ZERO entries with their (row, col) positions
# (0, 5)  → row 0, column 5 = value 1  (P6)
# (0, 6)  → row 0, column 6 = value 1  (P7)
# (0, 8)  → row 0, column 8 = value 2  (P9)


Sparse CSR Matrix:
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3 stored elements and shape (1, 9)>
  Coords	Values
  (0, 5)	1
  (0, 6)	1
  (0, 8)	2


## 6. 🔍 Reading the Sparse Matrix Output

When you `print()` a sparse matrix, you see:

```
  (0, 5)    1
  (0, 6)    1
  (0, 8)    2
```

This means:
| Output | Meaning |
|--------|---------|
| `(0, 5)` | Row 0, Column 5 |
| `1` | Value at that position |

> It **hides all zeros** — you only see what matters!
> Normal print of this 9-element array would show: `[0 0 0 0 0 1 1 0 2]`
> Sparse print shows only: positions with non-zero values → saves memory!


## 7. 🛠️ Useful Methods on Sparse Matrix

### 7a. `.data` — View Non-Zero Values


In [4]:
sparse_purchase = csr_matrix(np.array([0, 0, 0, 0, 0, 1, 1, 0, 2]))

# .data gives array of non-zero values only
print("Non-zero values (.data):", sparse_purchase.data)
# [1 1 2] → only the actual purchased quantities


Non-zero values (.data): [1 1 2]


### 7b. `.count_nonzero()` — Count Non-Zero Elements


In [5]:
count = sparse_purchase.count_nonzero()
print("Number of non-zero values:", count)
# Output: 3  → user bought from 3 different products

total_elements = sparse_purchase.shape[1]  # total products
print("Total products:", total_elements)
print("Sparsity:", round((1 - count/total_elements) * 100, 1), "%")
# Higher % = more sparse (more zeros)


Number of non-zero values: 3
Total products: 9
Sparsity: 66.7 %


### 7c. `.eliminate_zeros()` — Remove Stored Zeros


In [6]:
# Sometimes a matrix stores 0 explicitly (happens after operations)
# eliminate_zeros() removes those "stored zeros"

import numpy as np
from scipy.sparse import csr_matrix

# Create matrix with explicit 0 stored (simulate by modifying)
arr = np.array([3, 0, 0, 7, 0, 5])
sparse_arr = csr_matrix(arr)

print("Before eliminate_zeros - data:", sparse_arr.data)

# Now manually zero out one value (simulating what might happen in computation)
sparse_arr.data[0] = 0  # set first stored value to 0

print("After manually zeroing - data:", sparse_arr.data)

sparse_arr.eliminate_zeros()   # removes the stored 0
print("After eliminate_zeros - data:", sparse_arr.data)


Before eliminate_zeros - data: [3 7 5]
After manually zeroing - data: [0 7 5]
After eliminate_zeros - data: [7 5]


### 7d. `.sum_duplicates()` — Merge Repeated Positions


In [7]:
from scipy.sparse import coo_matrix

# COO format allows duplicate (row, col) entries
# This simulates a situation where same position is updated twice
row_idx = np.array([0, 0, 0])   # all in row 0
col_idx = np.array([2, 2, 4])   # col 2 appears TWICE (duplicate)
values  = np.array([10, 5, 7])  # 10 + 5 should merge at col 2

sparse_coo = coo_matrix((values, (row_idx, col_idx)), shape=(1, 6))
sparse_csr = sparse_coo.tocsr()   # convert to CSR first

print("Before sum_duplicates - data:", sparse_csr.data)
# The two values at col 2 may still be separate

sparse_csr.sum_duplicates()      # merge: 10+5 = 15 at col 2
print("After sum_duplicates - data:", sparse_csr.data)
# col 2 = 15, col 4 = 7


Before sum_duplicates - data: [15  7]
After sum_duplicates - data: [15  7]


### 7e. `.tocsc()` — Convert CSR to CSC


In [8]:
sparse_csr_mat = csr_matrix(np.array([0, 3, 0, 0, 7, 0, 2]))

print("Original format:", type(sparse_csr_mat))

sparse_csc_mat = sparse_csr_mat.tocsc()   # convert to CSC format
print("After tocsc():", type(sparse_csc_mat))
print("CSC data:", sparse_csc_mat.data)
# Same data, just stored differently (column-wise instead of row-wise)


Original format: <class 'scipy.sparse._csr.csr_matrix'>
After tocsc(): <class 'scipy.sparse._csc.csc_matrix'>
CSC data: [3 7 2]


## 8. 🏨 Real-Life Example: Hotel Room Booking Matrix

> **Scenario:** A hotel has 5 rooms and 10 days. We track if a room is booked (1) or not (0).
> Most days, most rooms are empty → **sparse data!**


In [9]:
from scipy.sparse import csr_matrix
import numpy as np

# Rows = Rooms (5), Columns = Days (10)
# 1 = booked, 0 = empty
hotel_booking = np.array([
    [0, 1, 0, 0, 0, 0, 0, 1, 0, 0],  # Room 1: booked day 2, 8
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],  # Room 2: booked day 4
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # Room 3: never booked
    [1, 0, 0, 0, 0, 1, 0, 0, 0, 0],  # Room 4: booked day 1, 6
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],  # Room 5: booked day 9
])

print("Normal matrix shape:", hotel_booking.shape, "= 50 elements stored")
print()

sparse_hotel = csr_matrix(hotel_booking)
print("Sparse Matrix (only bookings shown):")
print(sparse_hotel)


Normal matrix shape: (5, 10) = 50 elements stored

Sparse Matrix (only bookings shown):
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 6 stored elements and shape (5, 10)>
  Coords	Values
  (0, 1)	1
  (0, 7)	1
  (1, 3)	1
  (3, 0)	1
  (3, 5)	1
  (4, 8)	1


In [10]:
# How many bookings? (non-zero = booked)
total_booked = sparse_hotel.count_nonzero()
total_slots = hotel_booking.shape[0] * hotel_booking.shape[1]

print(f"Total slots: {total_slots}")
print(f"Booked slots: {total_booked}")
print(f"Empty slots: {total_slots - total_booked}")
print(f"Occupancy rate: {total_booked/total_slots*100:.1f}%")
print(f"Sparsity: {(1 - total_booked/total_slots)*100:.1f}%")

print("Actual booking values stored:", sparse_hotel.data)


Total slots: 50
Booked slots: 6
Empty slots: 44
Occupancy rate: 12.0%
Sparsity: 88.0%
Actual booking values stored: [1 1 1 1 1 1]


## 9. 🛒 Real-Life Example 2: E-Commerce User-Product Matrix

> **Scenario:** 1000 users, 500 products. Each user buys only 2-3 items.
> This matrix would be ~99.5% zeros → classic sparse problem in recommendation systems!


In [11]:
import numpy as np
from scipy.sparse import csr_matrix

# Simulate small version: 6 users, 10 products
# 1 = purchased
user_product = np.array([
    [0, 0, 1, 0, 0, 0, 0, 0, 0, 1],  # User 0: bought P2, P9
    [0, 1, 0, 0, 0, 0, 1, 0, 0, 0],  # User 1: bought P1, P6
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],  # User 2: bought nothing
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0],  # User 3: bought P0, P8
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],  # User 4: bought P4
    [0, 0, 0, 1, 0, 0, 0, 1, 0, 0],  # User 5: bought P3, P7
])

sparse_up = csr_matrix(user_product)

print("Sparse user-product matrix:")
print(sparse_up)
print(f"Total purchases recorded: {sparse_up.count_nonzero()}")
print(f"Total slots: {user_product.shape[0] * user_product.shape[1]}")

# This is what recommendation systems (Netflix, Amazon) use!
# scikit-learn's TF-IDF and CountVectorizer output exactly this kind of matrix


Sparse user-product matrix:
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 9 stored elements and shape (6, 10)>
  Coords	Values
  (0, 2)	1
  (0, 9)	1
  (1, 1)	1
  (1, 6)	1
  (3, 0)	1
  (3, 8)	1
  (4, 4)	1
  (5, 3)	1
  (5, 7)	1
Total purchases recorded: 9
Total slots: 60


## 10. 🔗 Connection to Machine Learning

> **Important:** scikit-learn (the ML library you will use) works with sparse matrices all the time!

| ML Task | Where Sparse Appears |
|---------|---------------------|
| Text Classification | Each document = row, each word = column, value = count |
| Recommendation Systems | User-item matrix (most values 0) |
| Feature Engineering | One-hot encoding creates many zeros |
| Logistic Regression | sklearn accepts sparse input directly |

> When you do `from sklearn.feature_extraction.text import TfidfVectorizer`,
> the output `X` is a **CSR sparse matrix** — now you know what that means! 🎉


In [12]:
# Mini demo: how sklearn uses sparse internally
# (you'll learn this fully in ML section)
from scipy.sparse import csr_matrix
import numpy as np

# Imagine this is text data: 3 documents, 5 words
# Value = how many times word appears
text_matrix = np.array([
    [0, 0, 3, 1, 0],   # Doc 1: uses word3 and word4
    [2, 0, 0, 0, 1],   # Doc 2: uses word1 and word5
    [0, 1, 0, 0, 0],   # Doc 3: uses only word2
])

sparse_text = csr_matrix(text_matrix)
print("Text feature matrix (sparse):")
print(sparse_text)

print("Only non-zero values (word occurrences):", sparse_text.data)
# sklearn TfidfVectorizer returns exactly this kind of matrix!


Text feature matrix (sparse):
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 5 stored elements and shape (3, 5)>
  Coords	Values
  (0, 2)	3
  (0, 3)	1
  (1, 0)	2
  (1, 4)	1
  (2, 1)	1
Only non-zero values (word occurrences): [3 1 2 1 1]


## 11. 🔄 Converting Between Sparse Formats


In [13]:
from scipy.sparse import csr_matrix, csc_matrix
import numpy as np

arr = np.array([
    [0, 0, 5, 0],
    [3, 0, 0, 0],
    [0, 0, 0, 7],
])

# Start as CSR
csr = csr_matrix(arr)
print("CSR format:")
print(csr)
print("Type:", type(csr))

# Convert to CSC
csc = csr.tocsc()
print("CSC format (same data, column-wise storage):")
print(csc)
print("Type:", type(csc))

# Convert back to dense (normal numpy array)
dense_again = csr.toarray()
print("Back to dense array:")
print(dense_again)


CSR format:
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3 stored elements and shape (3, 4)>
  Coords	Values
  (0, 2)	5
  (1, 0)	3
  (2, 3)	7
Type: <class 'scipy.sparse._csr.csr_matrix'>
CSC format (same data, column-wise storage):
<Compressed Sparse Column sparse matrix of dtype 'int64'
	with 3 stored elements and shape (3, 4)>
  Coords	Values
  (1, 0)	3
  (0, 2)	5
  (2, 3)	7
Type: <class 'scipy.sparse._csc.csc_matrix'>
Back to dense array:
[[0 0 5 0]
 [3 0 0 0]
 [0 0 0 7]]


## 12. ⚠️ Common Mistakes

| Mistake | Problem | Fix |
|---------|---------|-----|
| `csr_matrix([[1,0],[0,2]]).data` → confused about result | `.data` only shows non-zeros | Remember: zeros are hidden |
| Printing large sparse matrix | Shows thousands of lines | Use `.toarray()` only for small matrices |
| Forgetting to call `.eliminate_zeros()` | Stored zeros waste space after computations | Always clean up after matrix operations |
| Using CSR for column operations | Slow | Use CSC for column-heavy work |
| `.count_nonzero()` after operations | May count stored zeros too | Call `.eliminate_zeros()` first |



In [14]:
# Mistake demo: .data vs .toarray()
from scipy.sparse import csr_matrix
import numpy as np

m = csr_matrix(np.array([1, 0, 0, 5, 0, 3]))

print("WRONG way to see all values → .data only gives non-zeros:")
print(m.data)   # [1, 5, 3] — zeros missing!

print("CORRECT way to see ALL values → .toarray():")
print(m.toarray())  # [[1, 0, 0, 5, 0, 3]] — full array


WRONG way to see all values → .data only gives non-zeros:
[1 5 3]
CORRECT way to see ALL values → .toarray():
[[1 0 0 5 0 3]]


## 13. 📋 Cheat Sheet

```python
from scipy.sparse import csr_matrix, csc_matrix

# Create
sparse = csr_matrix(array)             # from numpy array
sparse = csr_matrix((data, (row, col)), shape=(m, n))  # from (data, indices)

# Inspect
sparse.data                            # non-zero values
sparse.count_nonzero()                 # count of non-zeros
sparse.shape                           # (rows, cols)

# Clean up
sparse.eliminate_zeros()               # remove stored zeros (in-place)
sparse.sum_duplicates()                # merge duplicate positions (in-place)

# Convert
sparse.tocsc()                         # CSR → CSC
sparse.tocsr()                         # CSC → CSR
sparse.toarray()                       # sparse → dense numpy array

# Check type
type(sparse)                           # <class 'scipy.sparse.csr.csr_matrix'>
```


## 14. 🎯 Interview Questions

**Q1. What is sparse data? Give a real example.**
A: Data where most values are 0. Example: a user-product matrix in e-commerce where 1 million users each buy only 2-3 of 500,000 products → 99.99% zeros.

**Q2. What is the difference between CSR and CSC?**
A: CSR (Compressed Sparse Row) is efficient for row operations and matrix-vector products. CSC (Compressed Sparse Column) is efficient for column operations. CSR is more commonly used in ML.

**Q3. Why use sparse matrices in ML?**
A: To save memory and speed up computation. E.g., text data with 50,000-word vocabulary but each document using only 200 words → 99.6% zeros. Storing all zeros wastes RAM.

**Q4. What does `.data` return in a sparse matrix?**
A: Only the non-zero values as a 1D NumPy array. Zero values are not stored at all.

**Q5. What is `.eliminate_zeros()` and when do you use it?**
A: Removes explicitly stored zero values from the sparse matrix (which can appear after arithmetic operations). Use it after matrix operations to keep the matrix clean and efficient.

**Q6. How does scikit-learn use sparse matrices?**
A: Functions like `TfidfVectorizer`, `CountVectorizer`, and `OneHotEncoder` return CSR sparse matrices. Algorithms like `LogisticRegression` and `SVM` accept sparse input directly — no need to convert to dense.

**Q7. What is sparsity % and how do you calculate it?**
A: `sparsity = (1 - count_nonzero / total_elements) * 100`. Higher % means more zeros and more benefit from sparse format.

**Q8. What is the difference between `.data` and `.toarray()`?**
A: `.data` returns only non-zero values (1D array). `.toarray()` returns the full matrix including zeros (2D dense array). Use `.toarray()` only for small matrices — large ones will crash RAM!


## 15. 💪 Practice Exercises

**Exercise 1 (Easy):**
Create a 1D array of 10 elements where only index 2, 5, and 8 have values (pick any non-zero values). Convert to CSR. Print `.data` and `.count_nonzero()`.

**Exercise 2 (Medium):**
Create a 4×5 2D array where at least 70% of values are 0. Convert to CSR. Calculate and print the sparsity percentage.

**Exercise 3 (Real-Life):**
You have 7 students and 10 quiz questions. Only 3 questions per student were attempted (rest = 0 = not attempted). Create this matrix and convert to CSR. Find out how many total questions were attempted across all students.

**Exercise 4 (Conversion):**
Take any sparse CSR matrix and convert it to CSC. Then convert back to dense with `.toarray()`. Verify both give same values.

**Exercise 5 (Challenge - Hotel Data Connection):**
Using your Hotel Booking Cancellation dataset:
- Create a simple matrix of [is_canceled] per [hotel type] per [month]
- Convert to CSR sparse
- Find sparsity % and count non-canceled bookings

---


## 📝 My Own Notes (Fill This In)

**What I understood well:**
*(write here)*

**What was confusing:**
*(write here)*

**New words I learned:**
- Sparse =
- CSR =
- CSC =

**Review Plan:**
| Date | What to Review |
|------|---------------|
| 2026-11-27 | Sections 1-5 (What is sparse, CSR creation, .data) |
| 2026-12-01 | Full notes + write hotel booking exercise from memory |
